# Tutorial 2: Dense BioModels Parameter Sweep (`MODEL1907260003`)

Estimated time: 30-50 minutes

## Prerequisites
- `libroadrunner` installed in the active notebook kernel environment.
- Network access for first-time model download from BioModels.

## Learning aims
- Primary package aim: run a real SBML model via the typed adapter workflow using the CLI from inside Jupyter.
- Secondary scientific aim: connect the `k_on` association-rate parameter to dynamic pathway behavior over time.

## Success criteria
- You generate and validate a dense `k_on` sweep spec.
- You attempt a full run and inspect/plot results from centralized sweep storage.
- You explain the observed trend in biological terms.


## Why this tutorial matters
This is your first tutorial where parameter scanning starts to look like an experiment design problem:
we use a denser and wider `k_on` grid than the quick 3-point scan, then summarize the full dynamic response as a heatmap.

The notebook also supports an offline-friendly path: if you already ran a BioModels example before,
it can reuse a local cached SBML file and avoid re-downloading from the network.


## Step 1: Build a dense sweep spec, then validate and plan

In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)

# --- Tutorial 2 setup: build a denser BioModels k_on sweep + seed the offline cache ---
import json
import shutil

import numpy as np

base_spec_path = root / "tutorials/specs/model.biomodels.quick.json"
dense_spec_path = root / "tmp/tutorials/specs/model.biomodels.tutorial2.dense.json"
spec_payload = json.loads(base_spec_path.read_text())

# Wider + denser than the original 3-point quick scan.
k_on_dense = [float(f"{v:.8g}") for v in np.linspace(8.0e-5, 1.2e-4, 11)]
spec_payload["design"]["grid"]["k_on"] = k_on_dense
spec_payload["storage"]["root"] = "tmp/tutorials/biomodels_store_dense"

dense_spec_path.parent.mkdir(parents=True, exist_ok=True)
dense_spec_path.write_text(json.dumps(spec_payload, indent=2, sort_keys=True))
DENSE_SPEC_REL = str(dense_spec_path.relative_to(root))

# Offline-friendly cache bootstrap: copy an already-downloaded SBML file if available.
biomodels_id = spec_payload["model"]["artifact"]["biomodels_id"]
target_cache = root / spec_payload["storage"]["root"] / "_cache" / "biomodels" / f"{biomodels_id}.xml"
if not target_cache.exists():
    # Search known cache locations (current store layout only; legacy paths removed).
    candidate_paths = [
        root / "tmp/tutorials/biomodels_store/_cache/biomodels" / f"{biomodels_id}.xml",
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            target_cache.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(candidate, target_cache)
            print(f"Seeded local SBML cache: {candidate} -> {target_cache}")
            break
    else:
        print("No local SBML cache found. First run may require network download from BioModels.")

print("Dense k_on sweep values:", k_on_dense)
run_mm_cli("validate", DENSE_SPEC_REL)
run_mm_cli("plan", DENSE_SPEC_REL)

## Step 2: Execute the dense model sweep

In [4]:
run_exit_code = run_mm_cli("run", DENSE_SPEC_REL, check=False)
if run_exit_code != 0:
    print("Run did not complete successfully. Continue to Step 3 for fallback guidance.")


Running point 1/3: {'k_on': 9e-05}
Running point 2/3: {'k_on': 0.0001}
Running point 3/3: {'k_on': 0.00011}
Stored sweep run: 9797f42513904df1ae23b1a37b8608fc
Run complete: 3 successful runs


## Step 3: Fallback path (if network/dependency blocks execution)

In [ ]:
if run_exit_code == 0:
    print("Dense run completed. Proceed to heatmap analysis.")
else:
    print("If run failed, this is usually because BioModels download or libroadrunner is unavailable.")
    print("You can still inspect the dense spec and planning output for onboarding.")
    print("Retry later in an environment with network + libroadrunner.")


## Step 4: Heatmap of dynamic response vs `k_on`

In [3]:
import csv
import json

import matplotlib.pyplot as plt
import numpy as np

registry_path = root / "tmp" / "run_registry.json"


def collect_dense_biomodel_sweeps() -> list[tuple[str, str, dict]]:
    if not registry_path.exists():
        return []
    registry = json.loads(registry_path.read_text())
    matches: list[tuple[str, str, dict]] = []
    for run_id, record_path in registry.items():
        candidate = Path(record_path)
        if not candidate.is_absolute():
            candidate = root / candidate
        if not candidate.exists():
            continue
        record = json.loads(candidate.read_text())
        sweep_rows_path = record.get("sweep_rows_path", "")
        if sweep_rows_path.startswith("tmp/tutorials/biomodels_store_dense/sweeps/"):
            matches.append((record.get("finished_at", ""), run_id, record))
    return matches


def parse_timeseries_from_row(row: dict[str, str]) -> tuple[np.ndarray, np.ndarray] | None:
    rows_blob = row.get("time_series__rows__json", "")
    if not rows_blob:
        return None

    col_items = []
    for key, value in row.items():
        if key.startswith("time_series__columns__"):
            try:
                idx = int(key.rsplit("__", 1)[1])
            except ValueError:
                continue
            col_items.append((idx, value))
    if not col_items:
        return None

    col_names = [name for _, name in sorted(col_items, key=lambda item: item[0])]
    table = json.loads(rows_blob)
    if not isinstance(table, list) or not table:
        return None

    matrix = np.array(
        [[float(entry.get(col, np.nan)) for col in col_names] for entry in table], dtype=float
    )
    if matrix.ndim != 2 or matrix.shape[1] == 0:
        return None

    first_col_is_time = col_names[0].lower() == "time" and np.all(np.diff(matrix[:, 0]) >= 0)
    if first_col_is_time:
        return matrix[:, 0], matrix[:, 1] if matrix.shape[1] > 1 else matrix[:, 0]
    return np.arange(matrix.shape[0], dtype=float), matrix[:, 0]


candidates = sorted(collect_dense_biomodel_sweeps(), reverse=True)
selected = None
series_by_k_on: list[tuple[float, np.ndarray, np.ndarray]] = []

for _, run_id, run_record in candidates:
    sweep_rows_path = root / run_record["sweep_rows_path"]
    rows = list(csv.DictReader(sweep_rows_path.read_text().splitlines()))
    rows = [row for row in rows if row.get("status") == "success"]

    trial_series: list[tuple[float, np.ndarray, np.ndarray]] = []
    for row in rows:
        parsed = parse_timeseries_from_row(row)
        if parsed is None:
            continue
        time, signal = parsed
        trial_series.append((float(row["k_on"]), time, signal))

    if trial_series:
        selected = (run_id, sweep_rows_path)
        series_by_k_on = trial_series
        break

if selected is None:
    print("No successful dense BioModels sweep with plottable time-series found yet.")
    print("Run Step 2 after dependencies/network (or local SBML cache) are available.")
else:
    run_id, sweep_rows_path = selected
    print("Using run id:", run_id)
    print("Sweep rows:", sweep_rows_path)

    series_by_k_on.sort(key=lambda item: item[0])
    min_len = min(len(item[2]) for item in series_by_k_on)
    k_vals = np.array([item[0] for item in series_by_k_on], dtype=float)
    time = series_by_k_on[0][1][:min_len]
    heat = np.vstack([item[2][:min_len] for item in series_by_k_on])

    plt.figure(figsize=(10, 4.5))
    im = plt.imshow(
        heat,
        origin="lower",
        aspect="auto",
        extent=[float(time[0]), float(time[-1]), float(k_vals[0]), float(k_vals[-1])],
        cmap="viridis",
    )
    plt.colorbar(im, label="state magnitude (first dynamic species column)")
    plt.title("Dense BioModels sweep: dynamic response over time vs k_on")
    plt.xlabel("time")
    plt.ylabel("k_on (1/(M*s))")
    plt.show()


No BioModels runs found yet. This is expected in offline/sandbox environments.


## Scientific checkpoint
`k_on` is an association-rate constant: larger values typically mean faster complex formation and earlier signaling engagement.

Write 3-4 sentences:
- In your heatmap, does increasing `k_on` shift high-activity regions earlier in time?
- Do higher `k_on` values mainly change timing, amplitude, or both?
- Which part of that pattern looks biologically plausible for ligand-receptor style kinetics?
